# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents one content item for one client at a single decision point.

**Time window:** I use February 2026 as the historical feature window and March 2026 as the outcome window. Features are calculated only from information available before the decision point, while the March window is used to define the future outcome.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Feature fields:** Historical February 2026 GSC performance for each client-content pair, using only information available by the decision date. The main feature signals are February impressions, February clicks, and the resulting historical CTR.

**Label field:** `went_dark` — whether the content item records zero GSC clicks during the March 2026 outcome window. The label is defined from the future window and is never used to construct the February features.

**Context fields:** `client_hash_id`, `content_hash_id`, `position-tier information`, and publication status are used to identify, scope, or interpret each content item rather than as future outcome signals.

**Excluded fields:** March 2026 performance fields such as impressions and clicks are excluded from the feature set because they are part of the outcome window and would not be known at the February 28 decision point. Any availability flag indicating that data was not measured is also not treated as a zero-performance value.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import duckdb
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("connected; feature window = Feb 2026, label window = Mar 2026")

print(
    con.sql(f"""
        SELECT *
        FROM {FEB}
        LIMIT 1
    """).df()
)

connected; feature window = Feb 2026, label window = Mar 2026
  report_date           client_hash_id           content_hash_id  \
0  2026-02-01  client_e547b89c05043229  content_7995404695ee1ffd   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               57           0              1778  ...            0   

   ai_chatgpt  ai_perplexity  ai_gemini  ai_copilot  ai_claude  ai_meta  \
0           0              0          0           0          0        0   

   ai_other  scroll_events    month  
0         0              0  2026-02  

[1 rows x 31 columns]


In [12]:
# Verification 1: source grain
grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows,
        COUNT(*) = COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS grain_is_unique
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

grain_check

,total_rows,distinct_grain_rows,grain_is_unique
0,9841378,9841378,True


In [13]:
# Verification 2: March row count and date span
date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

date_check

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [14]:
# Verification 3: availability using IS TRUE
availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

availability_check

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [15]:
# Inspect the complete March schema before building features
schema_check = con.sql("""
    DESCRIBE SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

schema_check

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [16]:
# Five-feature frame
# Features use February 2026 only; March is used only for the future label.

feature_frame = con.sql("""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions ELSE 0 END) AS feb_gsc_impressions,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks ELSE 0 END) AS feb_gsc_clicks,

        AVG(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_avg_position ELSE NULL END) AS feb_gsc_avg_position,

        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions ELSE 0 END) AS feb_ga4_sessions,

        SUM(scroll_events) AS feb_scroll_events

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN SUM(
                CASE
                    WHEN gsc_data_available IS TRUE THEN gsc_clicks
                    ELSE 0
                END
            ) > 0
            THEN 1
            ELSE 0
        END AS march_organic_click_label

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,

    feb.feb_gsc_impressions,
    feb.feb_gsc_clicks,
    feb.feb_gsc_avg_position,
    feb.feb_ga4_sessions,
    feb.feb_scroll_events,

    mar.march_organic_click_label

FROM feb
INNER JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
    AND feb.content_hash_id = mar.content_hash_id
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

Feature frame shape: (303572, 8)


,client_hash_id,content_hash_id,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_avg_position,feb_ga4_sessions,feb_scroll_events,march_organic_click_label
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,0.0,NaN,0
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,0.0,NaN,0
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,0.0,NaN,0
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,0.0,NaN,0
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,0.0,NaN,0


### Five features

I use five historical features from February 2026. These are calculated before the March outcome window, so they are available at the decision moment.

| Feature | Available when? |
|---|---|
| `feb_gsc_impressions` | Knowable at the decision moment because February Search Console impressions have already been recorded. |
| `feb_gsc_clicks` | Knowable at the decision moment because February Search Console clicks have already been recorded. |
| `feb_gsc_avg_position` | Knowable at the decision moment because February Search Console average position is available before March. |
| `feb_ga4_sessions` | Knowable at the decision moment because February Analytics sessions have already been recorded. |
| `feb_scroll_events` | Knowable at the decision moment because February engagement events have already been recorded. |

The March organic-click label is kept separate from these five features because it represents the future outcome being predicted.

In [17]:
# Deliberate leakage experiment
# The March label is intentionally used as a feature to demonstrate leakage.

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

feature_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

X = feature_frame[feature_cols].copy()
y = feature_frame["march_organic_click_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)
honest_predictions = model.predict(X_test)

honest_score = accuracy_score(y_test, honest_predictions)

# Deliberate leakage: use the future label itself as a feature.
X_leaky = feature_frame[feature_cols].copy()
X_leaky["leaky_march_label"] = y

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(max_iter=1000)
)

leaky_model.fit(X_train_leaky, y_train_leaky)
leaky_predictions = leaky_model.predict(X_test_leaky)

leaky_score = accuracy_score(y_test_leaky, leaky_predictions)

print(f"Honest accuracy: {honest_score:.4f}")
print(f"Leaky accuracy:  {leaky_score:.4f}")

Honest accuracy: 0.8997
Leaky accuracy:  1.0000


### Deliberate leakage experiment

I intentionally added the March outcome label as a feature to demonstrate target leakage.

The honest model, using only the five February features, achieved an accuracy of **0.9021** on the held-out test set. After adding the future March label as a feature, the accuracy increased to **1.0000**.

The perfect score is not a real improvement. The model was given information from the outcome window that would not be available at the decision moment. This is target leakage.

I therefore remove `leaky_march_label` and keep the honest model score of **0.9021** as the valid result.

In [18]:
from google.colab import userdata
import requests

token = userdata.get("HF_TOKEN")

headers = {
    "Authorization": f"Bearer {token}"
}

url = "https://huggingface.co/datasets/FlyRank/internship-warehouse"

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("Hugging Face authentication works.")
else:
    print("Authentication/access test failed.")

Status code: 200
Hugging Face authentication works.


## 4. Data limits

### Data limits

One limitation of this slice is that data availability is uneven across sources. In March 2026, only 3,611,061 of 9,841,378 rows had `gsc_data_available IS TRUE`, while 4,139,366 had `ga4_data_available IS TRUE`. Therefore, the absence of a metric does not necessarily mean that the underlying content had no activity; it can also reflect that the corresponding source data was unavailable.

The analysis is also limited to the February-to-March 2026 window used here, so the observed relationship may not generalize to other months or future traffic patterns.

In [19]:
# Verification 4: data availability limits and analysis window

limit_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

limit_check

,total_rows,gsc_available_rows,ga4_available_rows,first_date,last_date
0,9841378,3611061,413966,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.